# Discovery cohort preparation

Marianna Georgiou, Department of Psychiatry, Psychotherapy and Psychosomatics, Faculty of Medicine, RWTH Aachen, Aachen, Germany
**Email:** mariannag43@gmail.com
**Publication:**  Latent subtypes of longitudinal postpartum trajectories for identifying healthy women at risk for affective conditions

Prepare complete mood, stress, MPAS, and EPDS subscale trajectories, normalize feature blocks, export analysis tables, and evaluate k-means solutions using silhouettes and permutations.

Keep `trajectory_clustering_utils.py` and `cohort_data_utils.py` beside this notebook. 
## Inputs

- `files/inputs/combined_all_cohorts.xlsx`: master cohort data.
- `files/inputs/Subskalen Hauptstudie.xlsx`: discovery EPDS totals and subscales (`coh = [1, 2]`).
- Alternative inputs: `Subscales_sample3.xlsx` for `[3]`, or `Subskalen_allcohorts.xlsx` for the combined selection.

## Outputs

- `files/interm_files/full_subscales_trajectories_withoutnorm_{coh}.xlsx`: unnormalized features.
- `files/interm_files/sub_means_stds_{coh}.pkl`: normalization parameters.
- `files/interm_files/subscales_full_trajectories_{coh}.xlsx` and `.csv`: normalized features.
- `files/plots/sillouette_plotly_{coh}.png`: clustering evaluation plot.

`{coh}` becomes the cohort list, e.g. `[1, 2]`. 

## 1. Setup

In [1]:
import pandas as pd
import os
import trajectory_clustering_utils as hc
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import pickle


In [ ]:
from pathlib import Path

%load_ext dotenv
%dotenv
%load_ext autoreload
%autoreload 2

BASE_DIR = Path.cwd()
INPUT_PATH = BASE_DIR / "files" / "inputs"
OUTPUT_FILES = BASE_DIR / "files" / "interm_files"
OUTPUT_PLOTS = BASE_DIR / "files" / "plots"
OUTPUT_FILES.mkdir(parents=True, exist_ok=True)
OUTPUT_PLOTS.mkdir(parents=True, exist_ok=True)


In [ ]:
coh = [1, 2]


## 2. EPDS subscales

Harmonize columns and retain complete numeric records. The existing participant exclusion is preserved.

In [ ]:
if coh == [3]:
    subscales = pd.read_excel(INPUT_PATH / 'Subscales_sample3.xlsx')
elif coh == [1, 2]:
    subscales = pd.read_excel(INPUT_PATH / 'Subskalen Hauptstudie.xlsx')
    subscales = subscales[subscales['ID'] != 'BH_MS2']
else:
    subscales = pd.read_excel(INPUT_PATH / 'Subskalen_allcohorts.xlsx')             
subscales


In [ ]:
subscales.columns                              
subscales = subscales.iloc[:, :23]

if coh==[3]:
    rename_dict = {
        'EPDS T0 - Anamnesebogen! -': 'EPDStotal_t0',
        'Unnamed: 4': 'Anhedonie_t0',
        'Unnamed: 5': 'Anxiety_t0',
        'Unnamed: 6': 'Depression_t0',
        'EPDS T1': 'EPDStotal_t1',
        'Unnamed: 8': 'Anhedonie_t1',
        'Unnamed: 9': 'Anxiety_t1',
        'Unnamed: 10': 'Depression_t1',
        'EPDS T2': 'EPDStotal_t2',
        'Unnamed: 12': 'Anhedonie_t2',
        'Unnamed: 13': 'Anxiety_t2',
        'Unnamed: 14': 'Depression_t2',
        'EPDS T3': 'EPDStotal_t3',
        'Unnamed: 16': 'Anhedonie_t3',
        'Unnamed: 17': 'Anxiety_t3',
        'Unnamed: 18': 'Depression_t3',
        'EPDS T4': 'EPDStotal_t4',
        'Unnamed: 20': 'Anhedonie_t4',
        'Unnamed: 21': 'Anxiety_t4',
        'Unnamed: 22': 'Depression_t4'
    }
    subscales.rename(columns=rename_dict, inplace=True)

    subscales = subscales.drop(index = 0)

    subscales = subscales.reset_index(drop=True)
else:
        rename_dict = {
            'EPDS T0': 'EPDStotal_t0',
            'Unnamed: 4': 'Anhedonie_t0',
            'Unnamed: 5': 'Anxiety_t0',
            'Unnamed: 6': 'Depression_t0',
            'EPDS T1': 'EPDStotal_t1',
            'Unnamed: 8': 'Anhedonie_t1',
            'Unnamed: 9': 'Anxiety_t1',
            'Unnamed: 10': 'Depression_t1',
            'EPDS T2': 'EPDStotal_t2',
            'Unnamed: 12': 'Anhedonie_t2',
            'Unnamed: 13': 'Anxiety_t2',
            'Unnamed: 14': 'Depression_t2',
            'EPDS T3': 'EPDStotal_t3',
            'Unnamed: 16': 'Anhedonie_t3',
            'Unnamed: 17': 'Anxiety_t3',
            'Unnamed: 18': 'Depression_t3',
            'EPDS T4': 'EPDStotal_t4',
            'Unnamed: 20': 'Anhedonie_t4',
            'Unnamed: 21': 'Anxiety_t4',
            'Unnamed: 22': 'Depression_t4'
        }
        subscales.rename(columns=rename_dict, inplace=True)

        subscales = subscales.drop(index = 0)

        subscales = subscales.reset_index(drop=True)
subscales


In [ ]:
if coh == [3]:
    df_dropped = subscales.iloc[:, 3:]
    df_dropped = df_dropped.apply(pd.to_numeric, errors='coerce')
    df_dropped['ID'] = subscales['BH-Code']
    subscales_EPDS = df_dropped.dropna()
    subscales_EPDS
else:
    df_dropped = subscales.drop(columns=['VP-Nummer'])
    test = df_dropped[['EPDStotal_t0', 'Anhedonie_t0',
       'Anxiety_t0', 'Depression_t0', 'EPDStotal_t1', 'Anhedonie_t1',
       'Anxiety_t1', 'Depression_t1', 'EPDStotal_t2', 'Anhedonie_t2',
       'Anxiety_t2', 'Depression_t2', 'EPDStotal_t3', 'Anhedonie_t3',
       'Anxiety_t3', 'Depression_t3', 'EPDStotal_t4', 'Anhedonie_t4',
       'Anxiety_t4', 'Depression_t4']].apply(pd.to_numeric, errors='coerce')
    test["ID"] = df_dropped["ID"]
    print(df_dropped)
    subscales_EPDS = test.dropna()

subscales_EPDS


## 3. Cohort eligibility

Select cohorts, inspect missingness, and apply availability flags and ID/CODE requirements.

In [ ]:
combined_cohorts = pd.read_excel(INPUT_PATH / 'combined_all_cohorts.xlsx')
df = combined_cohorts[combined_cohorts['Cohort'].isin(coh)]

df


In [ ]:
from matplotlib.patches import Patch

common_ids = set(df["ID"]).intersection(df_dropped["ID"])
print(len(common_ids))

df_common = df_dropped[df_dropped["ID"].isin(common_ids)].copy()
print(len(df_common))

df_common = df_common.drop(columns=["Nummer"], errors="ignore")

df_common_drop = df_common.dropna()

print(len(df_common_drop))

df_common = df_common.reset_index(drop=True)

missing_matrix = df_common.isna()

plt.figure(figsize=(12, 8))

ax = sns.heatmap(
    ~missing_matrix,
    cmap=["blue", "yellow"],
    cbar=False,
    yticklabels=False
)

n_labels = 20

legend_elements = [
    Patch(facecolor="yellow", edgecolor="black", label="Existing value"),
    Patch(facecolor="blue", edgecolor="black", label="Missing value")
]

plt.legend(
    handles=legend_elements,
    loc="upper right",
    bbox_to_anchor=(1.25, 1)
)

plt.xlabel("Columns")
plt.ylabel("Participant Row Number")
plt.title(f"Missing vs Existing Values, Cohorts ({coh}), Discovery Cohort")

plt.show()


In [ ]:
df_cohort = hc.assert_variables(df)

mean_age = df_cohort['Age'].mean()
print(mean_age)

print(df_cohort['CODE'].value_counts())


## 4. Mood and stress trajectories

Use the first observed numeric score as baseline. Average subsequent observations over days 1–21, 22–42, 43–63, and 64–84; discard the final partial interval. Retain complete trajectories.

In [ ]:
def separate_baseline_from_traj(dataframe, choose_attribute):
                                 
    columns = [col for col in dataframe.columns if choose_attribute in col]
    scores = dataframe[columns].apply(pd.to_numeric, errors='coerce')

    first_non_nan_values = []
    first_non_nan_indices = []

    for _, row in scores.iterrows():
        non_nan = row.dropna()
        if not non_nan.empty:
            first_non_nan_values.append(non_nan.iloc[0])                     
            first_non_nan_indices.append(non_nan.index[0])                         
        else:
            first_non_nan_values.append(np.nan)
            first_non_nan_indices.append(None)

    df_baseline = pd.DataFrame({
        f'{choose_attribute} T0': first_non_nan_values,
        'CODE': dataframe['CODE'],
        'ID': dataframe['ID']
    })

    traj_rows = []
    for idx, start_col in enumerate(first_non_nan_indices):
        row_values = scores.iloc[idx].copy()
        if start_col is not None:
            start_pos = columns.index(start_col) + 1                        
            row_values.iloc[:start_pos] = np.nan                                                   
        traj_rows.append(row_values)

    df_trajectory = pd.DataFrame(traj_rows, columns=columns)
    df_trajectory['CODE'] = dataframe['CODE']
    df_trajectory['ID'] = dataframe['ID']

    return df_baseline, df_trajectory


In [ ]:
interval_size = 21
start_day = 1
finish_day = 88

stress = hc.select_mood_stress_values('Stress', df_cohort, interval_size, start_day, finish_day)                                     
df_baseline_stress, df_trajectory_stress = separate_baseline_from_traj(stress, 'Stress')                

mood = hc.select_mood_stress_values('Mood', df_cohort, interval_size, start_day, finish_day)
df_baseline_mood, df_trajectory_mood = separate_baseline_from_traj(mood, 'Mood')

long_df_stress, wide_df_stress = hc.clean_mood_stress('Stress', df_trajectory_stress, interval_size, start_day, finish_day)
long_df_mood, wide_df_mood = hc.clean_mood_stress("Mood",  df_trajectory_mood, interval_size, start_day, finish_day)

only_in_first_stress = set(df_trajectory_stress["ID"]) - set(wide_df_stress["ID"])
only_in_first_mood = set(df_trajectory_mood["ID"]) - set(wide_df_mood["ID"])
print(only_in_first_stress)
print(only_in_first_mood)

df_baseline_stress


In [ ]:
import re
                                                                        
old_columns = [col for col in wide_df_stress.columns if col.startswith('Stress T')]
print(old_columns)
                                                                          
new_columns = [f'Stress T{int(col.split("T")[1]) + 1}' for col in old_columns]

rename_dict = dict(zip(old_columns, new_columns))
wide_df_stress = wide_df_stress.rename(columns=rename_dict)

wide_df_stress


In [ ]:
old_columns = [col for col in wide_df_mood.columns if col.startswith('Mood T')]
print(old_columns)
                                                                          
new_columns = [f'Mood T{int(col.split("T")[1]) + 1}' for col in old_columns]

rename_dict = dict(zip(old_columns, new_columns))
wide_df_mood = wide_df_mood.rename(columns=rename_dict)
wide_df_mood


In [ ]:
df_baseline_stress = df_baseline_stress.dropna()
df_baseline_mood = df_baseline_mood.dropna()
wide_df_stress = wide_df_stress.dropna()
wide_df_mood = wide_df_mood.dropna()

merged_df_stress = pd.merge(df_baseline_stress, wide_df_stress, on=['CODE', 'ID'], how='inner')
merged_df_mood = pd.merge(df_baseline_mood, wide_df_mood, on=['CODE', 'ID'], how='inner')

merged_df_mood


In [ ]:
missing_stress = set(df["ID"]) - set(merged_df_stress["ID"])
missing_mood = set(df["ID"]) - set(merged_df_mood["ID"])

print(missing_stress, missing_mood)


## 5. MPAS and EPDS totals

Retain complete MPAS T1–T4 and EPDS T0–T4 records. Final features use EPDS subscales rather than totals.

In [ ]:
MPAS_parameters = ['MPAS T1', 'MPAS T2', 'MPAS T3', 'MPAS T4']

mpas = df_cohort[MPAS_parameters]
mpas = mpas.apply(pd.to_numeric, errors='coerce')
mpas['ID'] = df_cohort['ID']
mpas['CODE'] = df_cohort['CODE']

mpas_matrix = mpas.isna()

plt.figure(figsize=(8, 10))

ax = sns.heatmap(
    ~mpas_matrix,
    cmap=["blue", "yellow"],
    cbar=False,
    yticklabels=False
)

n_labels = 20
positions = np.linspace(0, len(mpas) - 1, n_labels, dtype=int)

ax.set_yticks(positions + 0.5)
ax.set_yticklabels(positions)

legend_elements = [
    Patch(facecolor="yellow", edgecolor="black", label="Existing value"),
    Patch(facecolor="blue", edgecolor="black", label="Missing value")
]

plt.legend(handles=legend_elements, loc="upper right", bbox_to_anchor=(1.3, 1))

plt.xlabel("MPAS parameters")
plt.ylabel("Participant number")
plt.title("MPAS Missing vs Existing Values")

plt.show()
mpas


In [ ]:
mpas = mpas.dropna()
MPAS_parameters = [ 'ID', "CODE", 'MPAS T1', 'MPAS T2', 'MPAS T3', 'MPAS T4']
wide_df_mpas = hc.mpas_epds_data_for_clustering(mpas, MPAS_parameters)
wide_df_mpas


In [ ]:
EPDS_parameters = ['EPDS T0', 'EPDS T1', 'EPDS T2', 'EPDS T3', 'EPDS T4']

epds = df_cohort[EPDS_parameters]
epds = epds.apply(pd.to_numeric, errors='coerce')
epds['ID'] = df_cohort['ID']
epds['CODE'] = df_cohort['CODE']
print(len(epds))
epds = epds.dropna()

EPDS_parameters = [ 'ID', "CODE",'EPDS T0','EPDS T1', 'EPDS T2', 'EPDS T3', 'EPDS T4']
wide_df_epds = hc.mpas_epds_data_for_clustering(epds, EPDS_parameters)
wide_df_epds


In [ ]:
sub_EPDS_parameters = ["EPDStotal_t0", "EPDStotal_t1", "EPDStotal_t2", "EPDStotal_t3", "EPDStotal_t4"]

total_epds = subscales_EPDS[sub_EPDS_parameters]
total_epds = total_epds.apply(pd.to_numeric, errors='coerce')

total_epds['ID'] = subscales_EPDS['ID']
total_epds = total_epds.dropna()

sub_EPDS_parameters = [ 'ID',"EPDStotal_t0", "EPDStotal_t1", "EPDStotal_t2", "EPDStotal_t3", "EPDStotal_t4"]
wide_df_epds_sub = hc.mpas_epds_data_for_clustering(total_epds, sub_EPDS_parameters)
              
wide_df_epds_sub


In [ ]:
coh


## 6. Align participants and export

Intersect available IDs, merge features, and export unnormalized data. IDs are assumed unique.

In [ ]:
combined_cohorts = pd.read_excel(INPUT_PATH / 'combined_all_cohorts.xlsx')
df_code = combined_cohorts[combined_cohorts['Cohort'].isin(coh)]

dfs = [merged_df_mood,merged_df_stress, wide_df_mpas, wide_df_epds_sub]
common_ids = set.intersection(set(wide_df_mood['ID']),set(wide_df_mpas["ID"]), set(wide_df_stress["ID"]), set(wide_df_epds_sub["ID"]))
print(len(common_ids))

epds_sub_f = subscales_EPDS[subscales_EPDS["ID"].isin(common_ids)]                                     
epds_subscales_f = wide_df_epds_sub[wide_df_epds_sub["ID"].isin(common_ids)]                     

stress_f = merged_df_stress[merged_df_stress['ID'].isin(common_ids)]
mood_f = merged_df_mood[merged_df_mood['ID'].isin(common_ids)]
epds_f = wide_df_epds[wide_df_epds['ID'].isin(common_ids)]                            
mpas_f = wide_df_mpas[wide_df_mpas['ID'].isin(common_ids)]
code = df_code[df_code['ID'].isin(common_ids)]
epds_f = epds_f.drop(columns=["CODE"])


In [ ]:
df_new = [stress_f, mood_f,mpas_f,epds_subscales_f, epds_sub_f]                                                                           
merged_new = pd.merge(df_new[0], df_new[1], on=['ID'], how='inner')
merged_new

for i in range(2, len(df_new)):
    merged_new = pd.merge(merged_new, df_new[i], on=['ID'], how='inner')
merged_new = merged_new.drop(columns=['EPDStotal_t0_x', 'EPDStotal_t1_x', 'EPDStotal_t2_x', 'EPDStotal_t3_x', 'EPDStotal_t4_x'])
merged_new["CODE"].value_counts()
merged_new


In [ ]:
output_path = OUTPUT_FILES / f'full_subscales_trajectories_withoutnorm_{coh}.xlsx'

merged_new.to_excel(output_path)
merged_new.columns


In [ ]:
coh


In [ ]:
custom_palette = {'PPD': '#FF7F50', 
                  'HC': '#0056A0', 
                  'AD': '#2C9F6D'}


## 7. Normalize feature blocks

Use one pooled mean and sample SD across numeric values per block. Numeric IDs also affect these moments; identifier dtypes matter. Save normalization parameters and normalized tables.

In [ ]:
anhedonia_sub = epds_sub_f.loc[:, epds_sub_f.columns.str.contains("Anhedonie")]
anhedonia_sub["ID"] = epds_sub_f["ID"]
anxiety_sub = epds_sub_f.loc[:, epds_sub_f.columns.str.contains("Anxiety")]
anxiety_sub["ID"] = epds_sub_f["ID"]
depression_sub = epds_sub_f.loc[:, epds_sub_f.columns.str.contains("Depression")]
depression_sub["ID"] = epds_sub_f["ID"]

norm_anhedonia, mean_anhedonia, std_anhedonia = hc.normalization(anhedonia_sub)
norm_anxiety, mean_anxiety, std_anxiety = hc.normalization(anxiety_sub)
norm_depression, mean_depression, std_depression = hc.normalization(depression_sub)
norm_anhedonia


In [ ]:
norm_mpas, mean_mpas, std_mpas = hc.normalization(mpas_f)

stress_columns = [col for col in stress_f.columns if 'Stress' in col]
stress_f[stress_columns] = stress_f[stress_columns].apply(pd.to_numeric, errors='coerce')
norm_stress, mean_stress, std_stress = hc.normalization(stress_f)

mood_columns = [col for col in mood_f.columns if 'Mood' in col]
mood_f[mood_columns] = mood_f[mood_columns].apply(pd.to_numeric, errors='coerce')
norm_mood , mean_mood, std_mood= hc.normalization(mood_f)

objects_to_save = {
    
    'mean_anh':mean_anhedonia,
    'std_anh':std_anhedonia,
    'mean_anx':mean_anxiety,
    'std_anx':std_anxiety,
    'mean_dep':mean_depression,
    'std_dep':std_depression,
    'mean_mpas': mean_mpas,
    'std_mpas':std_mpas,
    'mean_stress': mean_stress,
    'std_stress':std_stress,
    'mean_mood': mean_mood,
    'std_mood':std_mood

}
output_pkl = OUTPUT_FILES / f'sub_means_stds_{coh}.pkl'

with open(output_pkl, 'wb') as f:
    pickle.dump(objects_to_save, f)


In [ ]:
objects_to_save


In [ ]:
dfs = [ norm_mpas, norm_mood, norm_stress]            
merged_df = pd.merge(dfs[0], dfs[1], on=['ID', "CODE"], how='inner')

for i in range(2, len(dfs)):
    merged_df = pd.merge(merged_df, dfs[i], on=['ID', "CODE"], how='inner')
merged_df


In [ ]:
dfs_sub = [norm_anhedonia, norm_anxiety, norm_depression]

merged_df_sub = merged_df.copy()

for df_sub in dfs_sub:
    merged_df_sub = pd.merge(merged_df_sub, df_sub, on='ID', how='inner')

merged_df_sub.columns


In [ ]:
output_path = OUTPUT_FILES / f'subscales_full_trajectories_{coh}.xlsx'
merged_df_sub.to_excel(output_path)
merged_df_sub.to_csv(OUTPUT_FILES / f'subscales_full_trajectories_{coh}.csv', index=False)
merged_df_sub


## 8. Evaluate k-means clustering

Evaluate all six blocks together for k=2–6 with 10,000 shuffles per k. Columns are grouped by their first whitespace-delimited word; underscore-separated EPDS time points are shuffled separately. Refit k-means after shuffling. The plotted star at k=2 is fixed.

In [ ]:
merged_df_data = merged_df_sub.drop(columns=['ID', 'CODE'])
merged_df_sub.columns


In [ ]:
from scipy.cluster.vq import kmeans, vq
from sklearn.metrics import silhouette_samples


In [ ]:
import itertools

col_names = merged_df_data.columns.tolist()

anhedonia_columns = [col for col in col_names if 'Anhedonie' in col]
depression_columns = [col for col in col_names if 'Depression' in col]
anxiety_columns = [col for col in col_names if 'Anxiety' in col]

stress_columns = [col for col in col_names if 'Stress' in col]
mpas_columns = [col for col in col_names if 'MPAS' in col]
mood_columns = [col for col in col_names if 'Mood' in col]

col = [stress_columns,mood_columns,mpas_columns,anhedonia_columns, depression_columns, anxiety_columns]

combinations = []

for r in range(1, len(col) + 1):
    for combo in itertools.combinations(col, r):
        combinations.append(list(combo))
N_perm = 10000
all = [combinations[-1]]

p_values_results = {}
p_value_random_results = {}
silhouette_true = {}

df_trajectories = merged_df_sub

for combo in all:
    print(combo)
    all_columns = [col for sublist in combo for col in sublist]
    traj = df_trajectories[all_columns]
    df_cleaned = traj.replace([np.inf, -np.inf], np.nan).dropna()
    data_tra = np.array(df_cleaned.values, dtype = np.float32)

    p_values_tr = []
    p_value_random_tr = []
    silhouette_tr = []

    for n in range(2,7):
        centroids, _ = kmeans(data_tra, n)
        IDX, _ = vq(data_tra, centroids)

        s_true = silhouette_samples(data_tra, IDX)
        mean_s_true = np.mean(s_true)
        silhouette_tr.append(mean_s_true)

        sil_shuffled = np.zeros(N_perm)
                                              
        shuffled = np.zeros_like(data_tra)

        grouped_columns = {}
        for col in df_cleaned.columns:
            prefix = col.split()[0]                                               
            if prefix not in grouped_columns:
                grouped_columns[prefix] = []
            grouped_columns[prefix].append(col)

        for p in range(N_perm):
                                                                               
            current_col_index = 0
            for prefix, columns in grouped_columns.items():
                grouped_data = df_cleaned[columns].values
                shuffled_data = hc.shuffle(grouped_data)
                
                num_columns = len(columns)
                shuffled[:, current_col_index:current_col_index+num_columns] = shuffled_data
                
                current_col_index += num_columns

            centroids, _ = kmeans(shuffled, n)
            IDX_j, _ = vq(shuffled, centroids)
            sil_shuffled[p] = np.mean(silhouette_samples(shuffled, IDX_j))

        p_values_tr.append((np.sum(sil_shuffled > mean_s_true) + 1) / (N_perm + 1))                                                                                                                                                                      
        p_value_random_tr.append((np.sum(sil_shuffled[:-1] > sil_shuffled[-1]) + 1) / (N_perm + 1))

    key = '_'.join(all_columns)
    p_values_results[key] = p_values_tr
    p_value_random_results[key] = p_value_random_tr
    silhouette_true[key] = silhouette_tr


In [ ]:
p_values_tr


In [ ]:
mean_s_true


In [ ]:
def extract_words(input_string):
                                                 
    words = []

    target_words = ["Anxiety", "Depression", "Anhedonie", "MPAS", "Stress", "Mood"]

    for word in target_words:
                                                      
        if word in input_string:
                                      
            words.append(word)

    return words


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

for keys in p_values_results.keys():
    length = len(silhouette_true[keys])
    x = list(range(2, 2 + length))
    y1 = silhouette_true[keys]
    print(y1)
    y2 = p_values_results[keys]
    y3 = p_value_random_results[keys]
    features = extract_words(keys)

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Average Silhouette Scores", "Permutation Test p-values"),
        shared_xaxes=True
    )

    fig.add_trace(
        go.Scatter(x=x, y=y1, mode='lines+markers',
                   name='Silhouette', marker=dict(color="#254E70")),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(x=[x[0]], y=[y1[0]], mode='markers+text',
                   marker=dict(symbol="star", size=14, color="gold"),
                   text=["★"], textposition="top center", showlegend=False),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(x=x, y=y2, mode='lines+markers',
                   name='True vs. Shuffled', marker=dict(color="#8C271E")),
        row=1, col=2
    )
    fig.add_trace(
        go.Scatter(x=x, y=y3, mode='lines+markers',
                   name='Shuffled vs. Shuffled', marker=dict(color="#7f7f7f")),
        row=1, col=2
    )

    fig.update_layout(
        height=500, width=1000,
        title_text="Clustering Evaluation",
        title_x=0.5,
        legend=dict(
            orientation="h", yanchor="bottom", y=-0.2,
            xanchor="center", x=0.5
        ),
        template="plotly_white"
    )

    fig.update_xaxes(title_text="Number of Clusters", row=1, col=1)
    fig.update_xaxes(title_text="Number of Clusters", row=1, col=2)

    fig.update_yaxes(title_text="Silhouette Score", row=1, col=1)
    fig.update_yaxes(title_text="p-value", row=1, col=2)

    save_path = OUTPUT_PLOTS / f'sillouette_plotly_{coh}.png'
    fig.write_image(save_path, scale=3)
    print(f"Saved Plotly figure to {save_path}")

    fig.show()


## 9. Minimum detectable effect

Calculate Cohen’s d for a two-sided t-test at 80% power and alpha=0.05 using fixed group sizes: 281/49 (discovery) and 168/33 (validation).

In [ ]:
from statsmodels.stats.power import TTestIndPower

power_analysis = TTestIndPower()

mdes_discovery = power_analysis.solve_power(
    nobs1=281,
    ratio=49/281,
    alpha=0.05,
    power=0.80,
    alternative='two-sided'
)

mdes_validation = power_analysis.solve_power(
    nobs1=168,
    ratio=33/168,
    alpha=0.05,
    power=0.80,
    alternative='two-sided'
)

print(f"Discovery cohort MDES (Cohen's d): {mdes_discovery:.3f}")
print(f"Validation cohort MDES (Cohen's d): {mdes_validation:.3f}")
